In [ ]:
import pydicom
import numpy as np
import cv2
import os
from PIL import Image
import pylibjpeg

input_root = "./data"
output_root = "./output_images"
output_format = "png"  # 또는 'jpg'

# 대상 폴더 이름 목록
target_folders = ["pre indx"]

for root, _, files in os.walk(input_root):
    if os.path.basename(root).lower() not in target_folders:
        continue

    for file in files:
        if file.lower().endswith(".dcm"):
            dicom_path = os.path.join(root, file)

            try:
                ds = pydicom.dcmread(dicom_path)
                pixel_array = ds.pixel_array.astype(np.float32)

                img_norm = cv2.normalize(pixel_array, None, 0, 255, cv2.NORM_MINMAX)
                img_8bit = np.uint8(img_norm)

                # 저장 경로 만들기
                relative_path = os.path.relpath(dicom_path, input_root)
                save_path = os.path.splitext(relative_path)[0] + f".{output_format}"
                full_save_path = os.path.join(output_root, save_path)

                os.makedirs(os.path.dirname(full_save_path), exist_ok=True)
                img_rgb = Image.fromarray(img_8bit)
                img_rgb.save(full_save_path)

                print(f"✅ Saved: {full_save_path}")
            except Exception as e:
                print(f"❌ Failed to convert {dicom_path} — {e}")


In [ ]:
import pydicom
import numpy as np
import cv2
import os
from PIL import Image
import pylibjpeg

input_root = "./data"
output_root = "./output_images"
output_format = "png"  # 또는 'jpg'

target_folders = ["pre_indx"]

# (환자 번호: 변환할 인덱스 리스트)
target_dict = {
    4: [34],
    5: [42, 43, 44],
    9: [36],
    21: [43]
}

for root, _, files in os.walk(input_root):
    folder_name = os.path.basename(root)
    if folder_name.lower() not in target_folders:
        continue

    # 환자번호 추출 (폴더명 예: '4_206186867 문정연/pre_indx')
    try:
        patient_base = root.split(os.sep)
        # 환자 폴더 찾기 ('4_206...' 형식)
        patient_id_str = [seg for seg in patient_base if '_' in seg and seg.split('_')[0].isdigit()][0]
        patient_id = int(patient_id_str.split('_')[0])
    except Exception:
        continue

    if patient_id not in target_dict:
        continue

    for file in files:
        if not file.lower().endswith(".dcm"):
            continue

        # 슬라이스 인덱스(예: '34-dicom-00..dcm' → 34 추출)
        slice_prefix = file.split('-')[0]
        if not (slice_prefix.isdigit() and int(slice_prefix) in target_dict[patient_id]):
            continue

        dicom_path = os.path.join(root, file)

        try:
            ds = pydicom.dcmread(dicom_path)
            pixel_array = ds.pixel_array.astype(np.float32)
            img_norm = cv2.normalize(pixel_array, None, 0, 255, cv2.NORM_MINMAX)
            img_8bit = np.uint8(img_norm)
            # 저장 경로 만들기
            relative_path = os.path.relpath(dicom_path, input_root)
            save_path = os.path.splitext(relative_path)[0] + f".{output_format}"
            full_save_path = os.path.join(output_root, save_path)
            os.makedirs(os.path.dirname(full_save_path), exist_ok=True)
            img_rgb = Image.fromarray(img_8bit)
            img_rgb.save(full_save_path)
            print(f"✅ Saved: {full_save_path}")
        except Exception as e:
            print(f"❌ Failed to convert {dicom_path} — {e}")
